# Task 6 hyperparameter sweeps

One cell per model. `run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 6 section)
- **nested settings** as `"outer.inner"`, e.g. `"temporal_biases.drift_regularization"`

Every run trains on the training split and is scored on the validation split: **MAE in days on every user's last gap (lower is better)**.
No refit on all data, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task6/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py`), the swept and constant values and the score of every run.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** there is no finalist check: Task 6 has one fixed validation split (every user's last gap). Sweeping `seed` (the fold assignment of the out-of-fold score) shows how much the score moves by chance.
`show_training_log=True` prints the training's own log lines of every run.

The score is the model's own out-of-fold MAE on the validation rows (as in `task6_ensemble.ipynb`).
The linear median model has no hyperparameters (no penalty) -> only its `seed` can be varied.

In [ ]:
%load_ext autoreload
%autoreload 2

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## LightGBM gap model (CPU)

In [ ]:
lightgbm_gap_sweep = run_grid_search("task6", "lightgbm_gap_model", {
    "lightgbm_num_leaves": [15, 31, 63],
    "lightgbm_min_child_samples": [50, 100, 200],
})

In [ ]:
# noise check: the same settings with other fold assignments
lightgbm_gap_seed_check = run_grid_search("task6", "lightgbm_gap_model", {
    "seed": [0, 1, 2],
})

## Overview

In [ ]:
print_sweep_overview("task6")